In [ ]:
pip install langchain langchain-core langchain-community langchain-google-genai pypdf langchain-huggingface faiss-cpu

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("research_paper.pdf")
pages = loader.load()

In [3]:
pages

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [ ]:
type(pages)

langchain_core.documents.base.Document

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20, length_function=len, is_separator_regex=False)

In [6]:
docs = splitter.split_documents(pages)

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
embeddings_model.embed_documents(
    [
        "Hi there!",
        "Oh, hello!",
        "What's your name?",
        "My friends call me World",
        "Hello World!"
    ]
)

[[0.04065167158842087,
  0.0386141799390316,
  -0.023672476410865784,
  0.044803690165281296,
  0.0672212764620781,
  -0.03028147481381893,
  0.002320279134437442,
  0.050436295568943024,
  0.04003792256116867,
  -0.008837752044200897,
  -0.009490743279457092,
  -0.028623519465327263,
  -0.013932785950601101,
  0.014657813124358654,
  0.008816380053758621,
  -0.02424011379480362,
  0.04949696734547615,
  -0.021830948069691658,
  -0.025835437700152397,
  0.011732684448361397,
  0.026901576668024063,
  0.027565743774175644,
  0.006897861137986183,
  0.055878713726997375,
  -0.00042818186921067536,
  0.04185905307531357,
  -0.023439787328243256,
  -0.014832474291324615,
  0.014646483585238457,
  0.0220074150711298,
  0.00845478754490614,
  -0.009681081399321556,
  0.015049003064632416,
  0.08467372506856918,
  2.0787490484508453e-06,
  0.029706496745347977,
  -0.009667342528700829,
  0.004832589998841286,
  0.01173438411206007,
  0.05281321331858635,
  -0.020353754982352257,
  0.035156138

In [9]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings_model)

In [10]:
retriever = vectorstore.as_retriever()
query = "Introduction"
response = retriever.get_relevant_documents(query)
print(response[0].page_content)

Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com


/tmp/ipython-input-2263735621.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = retriever.get_relevant_documents(query)


In [12]:
from google.colab import userdata
import os

api_key = userdata.get('GOOGLE_API_KEY')

os.environ["GOOGLE_API_KEY"] = api_key

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

In [35]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [36]:
prompt = """
You are a helpful assistant. Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:

"""

In [37]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(prompt)

In [38]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

chain = (
    RunnableParallel(
    {
        "context": retriever,          # fetches docs from retriever

        "question": RunnablePassthrough()  # passes user question directly
     }
    )
    | prompt_template
    | llm
    | parser
)

In [ ]:
response = chain.invoke("Introduction")
print(response)

The paper "Attention is All you Need" proposes a novel, simple network architecture for sequence transduction models that relies solely on an attention mechanism, entirely dispensing with recurrent and convolutional neural networks. While dominant models previously used complex RNNs or CNNs with attention, this new approach is shown to be superior in quality, more parallelizable, and requires significantly less training time. For instance, on English-to-German translation, their single model achieved 27.5 BLEU, improving over the best existing ensemble result by over 1 BLEU. On English-to-French translation, it achieved a BLEU score of 41.1, outperforming the previous single state-of-the-art model by 0.7 BLEU.


In [42]:
response = chain.invoke("Who is president of india")
print(response)

I'm sorry, but the provided context does not contain any information about the president of India. The documents are about a research paper titled "Attention is All you Need" and discuss neural network architectures for machine translation.
